# Top 10 Viral Moments Clipper (v2, landscape)

Paste a YouTube link (or point at a video in your Drive) and get the **10 strongest moments** as ranked, ready-to-upload **1920×1080 landscape clips**, each with a suggested title and an `.srt` caption file, saved to Google Drive.

**How moments are chosen.** Loudness alone can't tell a punchline from a cough, so every candidate is scored on several signals:

| Signal | What it catches |
|---|---|
| **Transcript** (Whisper) | hook phrases, questions, strong claims, numbers, emotional words, speaking pace |
| **Audience reactions** (YAMNet) | laughter, applause, cheering, shouting, gasps |
| **Energy** | moments clearly louder than the surrounding two minutes |
| **Visual pace** | scene cuts per minute |
| **Clean edges** | clips start at the start of a sentence and end at the end of one |
| **Claude** (optional) | reads each candidate's transcript and judges: strong hook, makes sense on its own, has a payoff. Can trim the edges and writes a title |

The scores are combined into one ranking. Clips never overlap.

**Run the cells top to bottom** (or *Runtime → Run all*). For transcription that's about 10× faster, switch to a GPU first: *Runtime → Change runtime type → T4 GPU*.

**Keep this browser tab open and your computer awake while it runs.** Free Colab drops sessions that look idle. If the connection drops anyway, reconnect and choose *Run all* again. The transcript is saved to your Drive as it's made, so the run picks up where it stopped instead of starting over.

## 1 · Setup
Run this once per session. Click **Allow** when Colab asks to connect Google Drive.

In [ ]:
#@title 1 · Setup (run once per session)
import os, sys, re, json, math, time, shutil, subprocess, hashlib, csv
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def run(cmd, check=True):
    """Run a command (list of args, no shell) and return the CompletedProcess."""
    res = subprocess.run([str(c) for c in cmd], capture_output=True, text=True)
    if check and res.returncode != 0:
        tail = (res.stderr or res.stdout or "").strip()[-1500:]
        raise RuntimeError(f"Command failed ({res.returncode}): {' '.join(map(str, cmd[:4]))} ...\n{tail}")
    return res


def run_stream(cmd):
    """Run a command, echo useful progress lines, and return (returncode, full log)."""
    proc = subprocess.Popen([str(c) for c in cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log, last_pct = [], -10
    for line in proc.stdout:
        log.append(line)
        m = re.search(r"\[download\]\s+([\d.]+)%", line)
        if m:
            pct = float(m.group(1))
            if pct - last_pct >= 10 or pct >= 100:
                print(line.strip())
                last_pct = pct
        elif line.startswith(("ERROR", "WARNING", "[Merger]")):
            print(line.rstrip())
    return proc.wait(), "".join(log)


def fmt_ts(t):
    t = max(0, int(t))
    return f"{t // 3600}:{t % 3600 // 60:02d}:{t % 60:02d}" if t >= 3600 else f"{t // 60}:{t % 60:02d}"


if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content")
    OUTPUT_ROOT = Path("/content/drive/MyDrive/viral_clips")
else:
    BASE = Path.cwd()
    OUTPUT_ROOT = BASE / "viral_clips"

print("Installing packages...")
run([sys.executable, "-m", "pip", "install", "-q", "-U", "yt-dlp[default]", "faster-whisper", "anthropic"])
if shutil.which("ffmpeg") is None:
    run(["apt-get", "-qq", "install", "-y", "ffmpeg"])

# yt-dlp needs a JavaScript runtime (deno) to read YouTube's video player.
DENO_BIN = Path.home() / ".deno" / "bin"
os.environ["PATH"] = f"{DENO_BIN}{os.pathsep}{os.environ['PATH']}"
if shutil.which("deno") is None:
    print("Installing deno (needed by yt-dlp for YouTube)...")
    run(["bash", "-c", "curl -fsSL https://deno.land/install.sh | sh -s -- -y"], check=False)

HAS_GPU = shutil.which("nvidia-smi") is not None and run(["nvidia-smi"], check=False).returncode == 0
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"GPU:  {'yes' if HAS_GPU else 'no  (Runtime > Change runtime type > T4 GPU makes transcription ~10x faster)'}")
print(f"deno: {'ok' if shutil.which('deno') else 'MISSING - YouTube downloads may fail; re-run this cell'}")
print(f"Clips will be saved under: {OUTPUT_ROOT}")
print("Setup complete.")

## 2 · Your video and settings
Paste your link into `VIDEO_URL`. **Re-run this cell every time you change a setting**, and again after any Colab restart or disconnect.

**Optional: Claude ranking.** Click the 🔑 *Secrets* icon in Colab's left sidebar, add a secret named `ANTHROPIC_API_KEY`, and turn on *Notebook access*. If there's no key, this step is skipped and the clips are ranked on the other signals alone.

In [ ]:
#@title 2 · Your video and settings
VIDEO_URL = "https://youtu.be/YuFs5RhfZnE"  #@param {type:"string"}
#@markdown Optional: use a video file already in your Drive instead of downloading, e.g. `/content/drive/MyDrive/myvideo.mp4`. Leave empty to download `VIDEO_URL`.
LOCAL_VIDEO_PATH = ""  #@param {type:"string"}

#@markdown ---
NUM_CLIPS = 10  #@param {type:"slider", min:1, max:20, step:1}
MIN_CLIP_SEC = 30  #@param {type:"integer"}
TARGET_CLIP_SEC = 90  #@param {type:"integer"}
MAX_CLIP_SEC = 180  #@param {type:"integer"}
#@markdown **pad** keeps the whole picture (black bars only if the source isn't 16:9). **crop** fills the 16:9 frame by trimming the edges.
FIT_MODE = "pad"  #@param ["pad", "crop"]
#@markdown Captions are always saved as `.srt` files, which you can upload to YouTube. Turn this on to also burn them into the video.
BURN_CAPTIONS = False  #@param {type:"boolean"}

#@markdown ---
#@markdown `auto` uses large-v3-turbo on a GPU, small on a CPU.
WHISPER_MODEL = "auto"  #@param ["auto", "large-v3-turbo", "large-v3", "medium", "small", "base"]
#@markdown Spoken language code such as `en` or `es`. Leave empty to auto-detect.
LANGUAGE = ""  #@param {type:"string"}
USE_REACTION_DETECTOR = True  #@param {type:"boolean"}
USE_SCENE_DETECTION = True  #@param {type:"boolean"}
USE_CLAUDE_RANKING = True  #@param {type:"boolean"}
CLAUDE_MODEL = "claude-opus-5-5"  #@param {type:"string"}
#@markdown Netscape-format cookies exported from a browser that's logged in to YouTube. Only needed if YouTube says *"Sign in to confirm you're not a bot"*.
COOKIES_PATH = "/content/cookies.txt"  #@param {type:"string"}

if not (5 <= MIN_CLIP_SEC <= TARGET_CLIP_SEC <= MAX_CLIP_SEC):
    raise ValueError("Clip lengths must satisfy 5 <= MIN_CLIP_SEC <= TARGET_CLIP_SEC <= MAX_CLIP_SEC")
print("Settings saved.")

## 3 · Get the video
Downloads up to 1080p. If it fails, this cell says why in plain English and what to do next.

**If YouTube says "Sign in to confirm you're not a bot":** YouTube blocks downloads from Google's cloud servers unless they carry a logged-in browser's cookies. This cell then shows an **upload button** for your `cookies.txt` file, along with step-by-step instructions, and retries automatically. The file is saved to `MyDrive/cookies.txt`, so later sessions find it on their own. Cookies expire after a few days or weeks; when they do, the cell asks for a new file.

In [ ]:
#@title 3 · Get the video
if "run_stream" not in globals():
    raise RuntimeError("Run Cell 1 (Setup) first.")
if "VIDEO_URL" not in globals():
    raise RuntimeError("Settings are missing - run Cell 2 first (Colab forgets them after a restart).")

DOWNLOAD_HINTS = {
    "not a bot": "YouTube is blocking this Colab server. Re-run this cell and upload a fresh cookies.txt when it asks (steps are printed above the upload button). If you already uploaded one, it has expired or was exported from the wrong window - export a new one.",
    "not a valid url": "The link isn't a valid video URL. Fix VIDEO_URL in Cell 2, re-run Cell 2, then this cell.",
    "unsupported url": "yt-dlp doesn't recognise this link. Use a normal video link such as https://youtu.be/VIDEO_ID.",
    "private video": "The video is private. Use a public or unlisted video, or cookies from an account that can watch it.",
    "video unavailable": "The video is unavailable (removed, private, or blocked in the server's region).",
    "requested format is not available": "YouTube offered no downloadable formats. This usually means deno is missing or there's a bot check: re-run Cell 1, add cookies, then retry.",
    "javascript runtime": "deno (the JavaScript runtime yt-dlp needs) is missing. Re-run Cell 1.",
    "members-only": "This is members-only content. Cookies from a member account are required.",
    "confirm your age": "The video is age-restricted. Cookies from a signed-in adult account are required.",
    "http error 429": "YouTube is rate-limiting this server. Wait a while, add cookies, or try Runtime > Disconnect and delete runtime to get a new server.",
    "unable to download webpage": "Couldn't reach YouTube from this server (network error). Check the link, then try Runtime > Disconnect and delete runtime to get a new server.",
    "http error 403": "YouTube refused the download (403). Add cookies, re-run Cell 1 to update yt-dlp, and retry.",
}


def explain_failure(log):
    found = [hint.format(cookies=COOKIES_PATH) for key, hint in DOWNLOAD_HINTS.items() if key in log.lower()]
    print("\n----- yt-dlp output (last lines) -----")
    print("\n".join(log.strip().splitlines()[-15:]))
    print("--------------------------------------")
    for hint in found or ["Read the yt-dlp output above. Re-running Cell 1 updates yt-dlp, which fixes many YouTube changes."]:
        print("-> " + hint)
    raise RuntimeError("Download failed - see the explanation above.")


if LOCAL_VIDEO_PATH.strip():
    SOURCE = Path(LOCAL_VIDEO_PATH.strip())
    if not SOURCE.exists():
        raise FileNotFoundError(f"LOCAL_VIDEO_PATH not found: {SOURCE}")
    key = hashlib.md5(f"{SOURCE.resolve()}:{SOURCE.stat().st_size}".encode()).hexdigest()[:12]
    WORK = BASE / "work" / key
    META = {"id": None, "title": SOURCE.stem, "webpage_url": None}
else:
    url = VIDEO_URL.strip()
    if not re.match(r"^https?://\S+$", url):
        raise ValueError(f"VIDEO_URL doesn't look like a link: {url!r}. Paste a full https:// link in Cell 2, re-run Cell 2, then this cell.")

    DRIVE_COOKIES = Path("/content/drive/MyDrive/cookies.txt")
    COOKIE_STEPS = """How to get your YouTube cookies file (takes about 2 minutes, on your computer):
  1. Install the 'Get cookies.txt LOCALLY' extension in Chrome or Edge, then open its
     extension settings and switch on 'Allow in Incognito' (InPrivate in Edge).
  2. Open a private/incognito window and sign in to YouTube there
     (a spare Google account is safer than your main one).
  3. In that same window go to  https://www.youtube.com/robots.txt
  4. Click the extension icon > Export. It saves a file like 'www.youtube.com_cookies.txt'.
  5. Close the private window straight away, so YouTube doesn't replace those cookies.
  6. Click 'Choose Files' below and pick that file."""

    def find_cookies():
        return next((p for p in [Path(COOKIES_PATH), DRIVE_COOKIES] if p.is_file()), None)

    def ask_for_cookies(reason):
        """Let the user upload cookies.txt right here; returns True if a usable file was saved."""
        if not IN_COLAB:
            return False
        from google.colab import files
        print(f"\n{reason}\n\n{COOKIE_STEPS}\n")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded.")
            return False
        data = next(iter(uploaded.values()))
        text = data.decode("utf-8", "ignore")
        if "youtube.com" not in text or "\t" not in text:
            print("That file doesn't look like a Netscape-format cookies.txt containing YouTube cookies.")
            return False
        Path(COOKIES_PATH).write_bytes(data)
        try:
            DRIVE_COOKIES.write_bytes(data)
            print(f"Saved. A copy is kept at {DRIVE_COOKIES} so future sessions use it automatically (delete it any time).")
        except OSError:
            print(f"Saved to {COOKIES_PATH}.")
        return True

    def ytdlp_cmd():
        cmd = [sys.executable, "-m", "yt_dlp", "--no-playlist", "--newline"]
        cf = find_cookies()
        return cmd + ["--cookies", str(cf)] if cf else cmd

    info_line = None
    for attempt in range(2):
        cf = find_cookies()
        print(f"Using cookies from {cf}" if cf else "No cookies file found - trying without cookies.")
        print("Looking up the video...")
        code_, log = run_stream(ytdlp_cmd() + ["--skip-download", "--print", "%(id)s\t%(title)s\t%(duration)s", url])
        info_line = next((l for l in log.splitlines() if l.count("\t") >= 2), None)
        if code_ == 0 and info_line:
            break
        if attempt == 0 and "not a bot" in log.lower():
            reason = ("YouTube wants proof you're a real viewer. Your cookies file was rejected - it has probably expired, so export a fresh one."
                      if cf else "YouTube wants proof you're a real viewer before it lets this server download.")
            if ask_for_cookies(reason):
                continue
        explain_failure(log)
    vid, title, _ = info_line.split("\t", 2)
    META = {"id": vid, "title": title, "webpage_url": f"https://www.youtube.com/watch?v={vid}" if re.fullmatch(r"[\w-]{11}", vid) else url}
    WORK = BASE / "work" / re.sub(r"[^\w-]", "_", vid)
    WORK.mkdir(parents=True, exist_ok=True)
    SOURCE = WORK / "source.mp4"
    print(f"Video: {title}")

    if SOURCE.exists():
        print("Already downloaded - reusing it.")
    else:
        print("Downloading (up to 1080p)...")
        code_, log = run_stream(ytdlp_cmd() + [
            "-f", "bv*[height<=1080]+ba/b[height<=1080]/bv*+ba/b",
            "--merge-output-format", "mp4",
            "--retries", "5", "--fragment-retries", "5",
            "-o", str(WORK / "source.%(ext)s"), url,
        ])
        if code_ != 0 or not SOURCE.exists():
            explain_failure(log)

WORK.mkdir(parents=True, exist_ok=True)
# Slow results (the transcript) are cached on Drive so they survive a Colab disconnect.
CACHE = OUTPUT_ROOT / "_cache" / WORK.name
CACHE.mkdir(parents=True, exist_ok=True)
print(f"Source ready: {SOURCE} ({SOURCE.stat().st_size / 1e6:.0f} MB)")

## 4 · Inspect the video and extract audio

In [ ]:
#@title 4 · Inspect the video and extract audio
import numpy as np

if "SOURCE" not in globals() or not Path(SOURCE).exists():
    raise RuntimeError("No source video - run Cell 3 first.")

probe = json.loads(run(["ffprobe", "-v", "error", "-print_format", "json", "-show_format", "-show_streams", SOURCE]).stdout)
DURATION = float(probe["format"]["duration"])
vstream = next((s for s in probe["streams"] if s["codec_type"] == "video"), None)
if vstream is None:
    raise RuntimeError("The source has no video stream.")
if not any(s["codec_type"] == "audio" for s in probe["streams"]):
    raise RuntimeError("The source has no audio track - this clipper needs audio to find moments.")
N_SEC = int(math.ceil(DURATION))
print(f"Duration {fmt_ts(DURATION)}  ·  {vstream['width']}x{vstream['height']}")

AUDIO_SR = 16000
raw_path = WORK / "audio16k.raw"
if not raw_path.exists():
    print("Extracting audio...")
    run(["ffmpeg", "-y", "-v", "error", "-i", SOURCE, "-vn", "-ac", "1", "-ar", AUDIO_SR, "-f", "s16le", raw_path])
AUDIO = np.fromfile(raw_path, dtype=np.int16).astype(np.float32) / 32768.0
print(f"Audio ready ({len(AUDIO) / AUDIO_SR:.0f}s).")

## 5 · Transcribe
Uses Whisper with word-level timestamps. On a T4 GPU, an hour of video takes about 2–4 minutes. On a CPU it can take as long as the video itself, so use the GPU if you can.

Progress is saved to Drive every minute. If Colab disconnects, re-run Cells 1–5 and it resumes from where it stopped.

In [ ]:
#@title 5 · Transcribe
if "AUDIO" not in globals() or "CACHE" not in globals():
    raise RuntimeError("Run Cells 3 and 4 first.")

import ctypes, glob, site, warnings, logging
warnings.filterwarnings("ignore", message=".*unauthenticated requests.*")
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)


def enable_cuda12_for_whisper():
    # faster-whisper's GPU engine (CTranslate2) is built against CUDA 12 (cuBLAS 12 + cuDNN 9).
    # Newer Colab GPU images ship a different CUDA version, which causes "libcublas.so.12 is not found".
    # Fix: install the CUDA 12 libraries from pip and preload them so the engine finds them.
    def find():
        roots = site.getsitepackages() + [site.getusersitepackages()]
        found = []
        for r in roots:
            found += glob.glob(f"{r}/nvidia/cublas/lib/libcublas*.so.12") + glob.glob(f"{r}/nvidia/cudnn/lib/libcudnn*.so.9")
        return sorted(set(found))

    libs = find()
    names = {Path(p).name for p in libs}
    if "libcublas.so.12" not in names or "libcudnn.so.9" not in names:
        print("Installing CUDA 12 libraries for GPU transcription (about 1 minute, once per session)...")
        run([sys.executable, "-m", "pip", "install", "-q", "nvidia-cublas-cu12", "nvidia-cudnn-cu12==9.*"])
        libs = find()
    pending = sorted(libs, key=lambda p: ("cublasLt" not in p, "libcudnn.so" not in p, p))
    for _ in range(4):  # the libraries depend on each other, so retry the ones that failed
        failed = []
        for p in pending:
            try:
                ctypes.CDLL(p, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                failed.append(p)
        pending = failed
        if not pending:
            break
    required = {"libcublas.so.12", "libcublasLt.so.12", "libcudnn.so.9", "libcudnn_ops.so.9", "libcudnn_cnn.so.9"}
    missing = required & {Path(p).name for p in pending}
    if missing:
        print(f"Couldn't load {sorted(missing)}.")
    return not missing


model_name = WHISPER_MODEL if WHISPER_MODEL != "auto" else ("large-v3-turbo" if HAS_GPU else "base")
cache = CACHE / f"transcript_{model_name}_{LANGUAGE or 'auto'}.json"
partial = CACHE / f"transcript_partial_{LANGUAGE or 'auto'}.json"


def transcribe(device, name, done):
    from faster_whisper import WhisperModel, BatchedInferencePipeline
    print(f"Loading Whisper '{name}' on {device.upper()}...")
    model = WhisperModel(name, device=device, compute_type="float16" if device == "cuda" else "int8")
    pipe = BatchedInferencePipeline(model=model)
    offset = done[-1]["end"] if done else 0.0
    if offset:
        print(f"Resuming from {fmt_ts(offset)} (earlier progress was saved).")
    seg_iter, info = pipe.transcribe(AUDIO[int(offset * AUDIO_SR):], language=LANGUAGE or None, word_timestamps=True,
                                     batch_size=16 if device == "cuda" else 4, beam_size=5 if device == "cuda" else 1)
    print(f"Language: {info.language} ({info.language_probability:.0%})")
    t_start, last_save, next_report = time.time(), time.time(), 0.05
    for seg in seg_iter:
        done.append({"start": seg.start + offset, "end": seg.end + offset, "text": seg.text.strip(),
                     "words": [[w.start + offset, w.end + offset, w.word.strip()] for w in (seg.words or [])]})
        if time.time() - last_save > 60:
            partial.write_text(json.dumps(done))
            last_save = time.time()
        frac = done[-1]["end"] / DURATION
        if frac >= next_report:
            rate = (done[-1]["end"] - offset) / max(time.time() - t_start, 1e-6)
            eta = (DURATION - done[-1]["end"]) / max(rate, 1e-6)
            print(f"  {frac:.0%} transcribed  ·  about {eta / 60:.0f} min left")
            next_report = frac + 0.05
    return done


if cache.exists():
    SEGMENTS = json.loads(cache.read_text())
    print(f"Loaded saved transcript ({len(SEGMENTS)} segments).")
else:
    t0 = time.time()
    done = json.loads(partial.read_text()) if partial.exists() else []
    device = "cpu"
    if HAS_GPU:
        try:
            device = "cuda" if enable_cuda12_for_whisper() else "cpu"
        except Exception as e:
            print(f"GPU setup failed ({e}).")
    if HAS_GPU and device == "cpu":
        print("Falling back to CPU - this will be slow.")
        model_name = "base" if WHISPER_MODEL == "auto" else model_name
    elif not HAS_GPU:
        print("No GPU - transcribing on CPU, which is slow. For ~10x speed: Runtime > Change runtime type > T4 GPU.")
    try:
        SEGMENTS = transcribe(device, model_name, done)
    except Exception as e:
        if device != "cuda":
            raise
        print(f"GPU transcription failed ({e}). Falling back to CPU with the 'base' model - this will be slow.")
        partial.write_text(json.dumps(done))
        SEGMENTS = transcribe("cpu", "base", done)
    SEGMENTS = [s for s in SEGMENTS if s["text"]]
    cache.write_text(json.dumps(SEGMENTS))
    partial.unlink(missing_ok=True)
    print(f"Transcribed {len(SEGMENTS)} segments in {(time.time() - t0) / 60:.1f} min.")

if not SEGMENTS:
    print("No speech found - moments will be picked from audio and visual signals only.")
else:
    for s in SEGMENTS[:3]:
        print(f"  [{fmt_ts(s['start'])}] {s['text']}")

## 6 · Audio signals: energy and audience reactions
Loudness is measured **relative to the surrounding two minutes**, so a moment counts as a spike only if it's louder than its own context. YAMNet, Google's audio-event model, flags laughter, applause, cheering and shouting.

In [ ]:
#@title 6 · Audio signals
if "AUDIO" not in globals():
    raise RuntimeError("Run Cell 4 first.")

# Loudness per second, in dB
padded = np.zeros(N_SEC * AUDIO_SR, dtype=np.float32)
padded[:min(len(AUDIO), len(padded))] = AUDIO[:len(padded)]
rms = np.sqrt(np.mean(padded.reshape(N_SEC, AUDIO_SR) ** 2, axis=1))
LOUD_DB = 20 * np.log10(rms + 1e-5)
# Floor near-silent seconds so pauses and silent intros don't make normal speech look like a spike
LOUD_DB = np.maximum(LOUD_DB, max(-60.0, float(np.percentile(LOUD_DB, 10))))

# Excitement = dB above the median of the surrounding ~2 minutes (capped at 20 dB)
half = min(60, max(1, N_SEC - 1))
ext = np.pad(LOUD_DB, half, mode="reflect")
local_median = np.median(np.lib.stride_tricks.sliding_window_view(ext, 2 * half + 1), axis=1)[:N_SEC]
EXCITE = np.clip(LOUD_DB - local_median, 0, 20)
print(f"Loudness scanned. Biggest spike: +{EXCITE.max():.1f} dB at {fmt_ts(EXCITE.argmax())}")

REACTION = np.zeros(N_SEC)
REACTION_LABEL = [""] * N_SEC
REACTION_CLASSES = {"Laughter", "Baby laughter", "Giggle", "Snicker", "Belly laugh", "Chuckle, chortle",
                    "Applause", "Clapping", "Cheering", "Crowd", "Screaming", "Shout", "Yell", "Whoop",
                    "Children shouting", "Gasp"}
if USE_REACTION_DETECTOR:
    try:
        try:
            import tensorflow_hub as hub
        except ImportError:  # missing, or a setuptools too new to provide pkg_resources
            run([sys.executable, "-m", "pip", "install", "-q", "tensorflow-hub", "setuptools<81"])
            for _m in [m for m in sys.modules if m.startswith(("tensorflow_hub", "pkg_resources"))]:
                del sys.modules[_m]
            import importlib
            importlib.invalidate_caches()
            import tensorflow_hub as hub
        import tensorflow as tf
        print("Loading YAMNet reaction detector...")
        yamnet = hub.load("https://tfhub.dev/google/yamnet/1")
        with tf.io.gfile.GFile(yamnet.class_map_path().numpy().decode()) as f:
            names = [row["display_name"] for row in csv.DictReader(f)]
        idx = [i for i, n in enumerate(names) if n in REACTION_CLASSES]
        chunk = 60 * AUDIO_SR
        for c0 in range(0, len(AUDIO), chunk):
            scores = yamnet(AUDIO[c0:c0 + chunk])[0].numpy()[:, idx]  # (frames, classes); frame hop 0.48 s
            for fi, row in enumerate(scores):
                sec = int(c0 / AUDIO_SR + fi * 0.48 + 0.48)
                if sec < N_SEC and row.max() > REACTION[sec]:
                    REACTION[sec] = float(row.max())
                    REACTION_LABEL[sec] = names[idx[int(row.argmax())]]
        hits = int((REACTION > 0.3).sum())
        print(f"Reaction scan done: {hits} seconds with likely laughter/applause/cheering.")
    except Exception as e:
        print(f"Reaction detector unavailable ({type(e).__name__}: {e}). Continuing without it.")
else:
    print("Reaction detector turned off in settings.")

## 7 · Visual pace (scene cuts)
Only keyframes are compared (video encoders put one at almost every cut), which is about 10× faster than decoding every frame. Turn off `USE_SCENE_DETECTION` in Cell 2 to skip this step.

In [ ]:
#@title 7 · Visual pace
CUTS = np.zeros(N_SEC)
if USE_SCENE_DETECTION:
    print("Scanning for scene cuts...")
    t0 = time.time()
    res = run(["ffmpeg", "-hide_banner", "-nostats", "-skip_frame", "nokey", "-i", SOURCE, "-an", "-sn",
               "-vf", "scale=320:-2,select='gt(scene,0.35)',metadata=print:file=-", "-f", "null", "-"], check=False)
    for m in re.finditer(r"pts_time:([\d.]+)", res.stdout):
        sec = int(float(m.group(1)))
        if sec < N_SEC:
            CUTS[sec] += 1
    print(f"Found {int(CUTS.sum())} scene cuts in {time.time() - t0:.0f}s.")
else:
    print("Scene detection turned off in settings.")

## 8 · Find and score candidate moments
Builds every candidate clip that starts and ends on a sentence boundary and is between `MIN_CLIP_SEC` and `MAX_CLIP_SEC` long. Each candidate is scored on all the signals, and the best non-overlapping ones are kept.

In [ ]:
#@title 8 · Find and score candidate moments
for _req in ("SEGMENTS", "EXCITE", "REACTION", "CUTS"):
    if _req not in globals():
        raise RuntimeError(f"Missing {_req} - run Cells 4-7 first.")

HOOK_RULES = [
    (r"\b(oh my (god|gosh)|no way|what the|holy|are you serious|i can'?t believe|wait,? what|wow)\b", 0.35),
    (r"\b(secret|truth|mistake|nobody|never|worst|best|crazy|insane|unbelievable|shocking|biggest|dangerous|illegal)\b", 0.2),
    (r"\b(how to|here'?s (how|why|what)|the (reason|problem|key|trick)|what if|you need to|stop|don'?t|why)\b", 0.15),
    (r"\b(love|hate|scared|afraid|angry|cried|crying|laugh|funny|hilarious|embarrass\w*|died|dead)\b", 0.15),
    (r"(\$\s?\d|\b\d[\d,.]*\s?(%|percent|k\b|million|billion|thousand|years?|times))", 0.15),
    (r"[!?]", 0.1),
]
SENT_END = (".", "?", "!", '"', "”")


def text_score(text):
    t = text.lower()
    return min(1.0, sum(w * min(2, len(re.findall(p, t))) for p, w in HOOK_RULES))


SPEECH, WPS, HOOKY = np.zeros(N_SEC), np.zeros(N_SEC), np.zeros(N_SEC)
for seg in SEGMENTS:
    seg["hook"] = text_score(seg["text"])
    a, b = int(seg["start"]), min(N_SEC, max(int(seg["start"]) + 1, int(math.ceil(seg["end"]))))
    rate = len(seg["text"].split()) / max(seg["end"] - seg["start"], 0.5)
    SPEECH[a:b] = 1
    WPS[a:b] = np.maximum(WPS[a:b], rate)
    HOOKY[a:b] = np.maximum(HOOKY[a:b], seg["hook"])

# ---- candidate windows -------------------------------------------------------------
cands = []
n = len(SEGMENTS)
for i in range(n):
    s_seg = SEGMENTS[i]
    clean_start = 1.0 if i == 0 or SEGMENTS[i - 1]["text"].rstrip().endswith(SENT_END) \
        or s_seg["start"] - SEGMENTS[i - 1]["end"] > 0.7 else 0.0
    last_len = -1e9
    for j in range(i, n):
        length = SEGMENTS[j]["end"] - s_seg["start"]
        if length > MAX_CLIP_SEC:
            break
        if length < MIN_CLIP_SEC or length - last_len < 6:
            continue
        clean_end = 1.0 if j == n - 1 or SEGMENTS[j]["text"].rstrip().endswith(SENT_END) \
            or SEGMENTS[j + 1]["start"] - SEGMENTS[j]["end"] > 0.7 else 0.0
        cands.append({"start": max(0.0, s_seg["start"] - 0.25), "end": min(DURATION, SEGMENTS[j]["end"] + 0.6),
                      "seg_i": i, "seg_j": j, "clean": (clean_start + clean_end) / 2})
        last_len = length

if not cands:  # little or no speech: fall back to a time grid
    lengths = sorted({min(L, DURATION) for L in (MIN_CLIP_SEC, TARGET_CLIP_SEC, MAX_CLIP_SEC)})
    for L in lengths:
        for s in np.arange(0, max(DURATION - L, 0) + 1e-6, 5.0):
            cands.append({"start": float(s), "end": float(min(DURATION, s + L)), "seg_i": None, "seg_j": None, "clean": 0.5})
print(f"{len(cands)} candidate windows.")

# ---- features ----------------------------------------------------------------------
span = max(TARGET_CLIP_SEC - MIN_CLIP_SEC, MAX_CLIP_SEC - TARGET_CLIP_SEC, 1)
for c in cands:
    a, b = int(c["start"]), max(int(c["start"]) + 1, min(N_SEC, int(math.ceil(c["end"]))))
    L = c["end"] - c["start"]
    sp, ex = SPEECH[a:b] > 0, EXCITE[a:b]
    top_q = max(1, len(ex) // 4)
    op = slice(a, min(b, a + 8))
    c["f"] = {
        "reaction": float(np.sort(REACTION[a:b])[-3:].mean()),
        "energy": float(np.sort(ex)[-top_q:].mean()),
        "dynamics": float(LOUD_DB[a:b].std()),
        "text": float((HOOKY[a:b][sp].mean() if sp.any() else 0) + 0.5 * HOOKY[a:b].max()),
        "opening": float(HOOKY[op].max() + EXCITE[op].mean() / 6 + REACTION[op].max()),
        "pace": float(WPS[a:b][sp].mean() if sp.any() else 0),
        "cuts": float(CUTS[a:b].sum() / L * 60),
        "dead_air": float(np.mean(~sp & (ex < 1.0))),
        "clean": c["clean"],
        "length_fit": float(max(0.0, 1 - abs(L - TARGET_CLIP_SEC) / span)),
    }

WEIGHTS = {"reaction": 0.22, "energy": 0.16, "text": 0.18, "opening": 0.12, "pace": 0.07, "dynamics": 0.07,
           "cuts": 0.05, "clean": 0.06, "length_fit": 0.10, "dead_air": -0.15}
ALREADY_0_1 = {"clean", "length_fit", "dead_air"}


def pct_rank(x):
    order = x.argsort().argsort()
    return order / max(len(x) - 1, 1)


if cands:
    total = np.zeros(len(cands))
    active = {}
    for k, w in WEIGHTS.items():
        vals = np.array([c["f"][k] for c in cands])
        if np.ptp(vals) == 0 and k not in ALREADY_0_1:
            continue  # signal missing or flat (e.g. detector off) -> ignore it
        active[k] = w
        total += w * (vals if k in ALREADY_0_1 else pct_rank(vals))
    total /= sum(abs(w) for w in active.values())
    for c, sc in zip(cands, total):
        c["signal"] = float(sc)
    print("Signals used: " + ", ".join(active))


def overlaps(a, b):
    return a["start"] < b["end"] and b["start"] < a["end"]


def pick_non_overlapping(items, k, key):
    chosen = []
    for c in sorted(items, key=key, reverse=True):
        if not any(overlaps(c, o) for o in chosen):
            chosen.append(c)
            if len(chosen) == k:
                break
    return chosen


def describe(c):
    """Short human-readable notes on why the signals liked this window."""
    a, b = int(c["start"]), max(int(c["start"]) + 1, min(N_SEC, int(math.ceil(c["end"]))))
    notes = []
    r = int(np.argmax(REACTION[a:b])) + a
    if REACTION[r] > 0.3:
        notes.append(f"{REACTION_LABEL[r].lower()} at {fmt_ts(r)}")
    e = int(np.argmax(EXCITE[a:b])) + a
    if EXCITE[e] > 4:
        notes.append(f"loudness spike +{EXCITE[e]:.0f} dB at {fmt_ts(e)}")
    segs = clip_segments(c)
    if segs:
        best = max(segs, key=lambda s: s["hook"])
        if best["hook"] >= 0.3:
            notes.append(f'hook line: "{best["text"][:80]}"')
    if c["f"]["cuts"] > 12:
        notes.append(f"fast visual pace ({c['f']['cuts']:.0f} cuts/min)")
    return "; ".join(notes) or "steady segment"


def clip_segments(c):
    return [s for s in SEGMENTS if s["end"] > c["start"] + 0.1 and s["start"] < c["end"] - 0.1]


def auto_title(c):
    segs = clip_segments(c)
    if not segs:
        return f"Moment at {fmt_ts(c['start'])}"
    best = max(segs, key=lambda s: (s["hook"], len(s["text"])))
    t = re.sub(r"\s+", " ", best["text"]).strip(" .,-")
    return t if len(t) <= 70 else t[:68].rsplit(" ", 1)[0].rstrip(",;:") + "..."


POOL_SIZE = NUM_CLIPS * 3 if USE_CLAUDE_RANKING else NUM_CLIPS
POOL = pick_non_overlapping(cands, POOL_SIZE, key=lambda c: c["signal"])
for c in POOL:
    c["why"] = describe(c)
    c["title"] = auto_title(c)
if len(POOL) < NUM_CLIPS:
    print(f"Note: only {len(POOL)} non-overlapping moments fit in this video with MIN_CLIP_SEC={MIN_CLIP_SEC}.")
print(f"\nTop candidates by signal score ({len(POOL)} kept):")
for c in sorted(POOL, key=lambda c: -c["signal"])[:NUM_CLIPS]:
    print(f"  {c['signal']:.2f}  {fmt_ts(c['start'])}-{fmt_ts(c['end'])}  {c['why']}")

## 9 · Rank with Claude (optional)
Claude reads the transcript of each shortlisted candidate and scores how well it works as a stand-alone YouTube clip. It looks for a strong opening hook, a clip that makes sense without the rest of the video, and a payoff. It can tighten the start and end, and it writes a title.

The final score is **35% signals + 65% Claude**. If there's no API key, the signal ranking is used as-is.

In [ ]:
#@title 9 · Rank with Claude (optional)
if "POOL" not in globals():
    raise RuntimeError("Run Cell 8 first.")

for c in POOL:
    c.pop("claude", None)
    c["final"] = c["signal"]


def get_api_key():
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key and IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            key = None
    return key


SYSTEM_PROMPT = """You are a senior YouTube editor choosing which moments of a long video to publish as stand-alone landscape clips (30 seconds to 3 minutes) on a normal YouTube channel.

Judge each candidate on:
- Hook: do the first few seconds make a viewer want to keep watching?
- Stand-alone: does it make sense without the rest of the video (no unresolved "as I said earlier", no missing setup)?
- Payoff: does it deliver a punchline, insight, reveal, strong emotion, or satisfying conclusion before it ends?
- Shareability: is it funny, surprising, useful, controversial, or quotable enough that people would share or comment?
- Clean edges: does it start at the beginning of a thought and end after the thought is finished?

Use the whole 0-100 range and be discriminating: most candidates should land between 30 and 70, and only genuinely strong clips above 80.
You may tighten a clip by choosing a later start and/or an earlier end, both inside the candidate's range, when that gives a stronger hook or cleaner ending. Never extend beyond the candidate's range. Otherwise return the original start and end.
Titles: under 70 characters, specific and curiosity-provoking, faithful to what actually happens (no misleading clickbait), and in the same language as the transcript."""

CLAUDE_SCHEMA = {
    "type": "object",
    "properties": {
        "clips": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "id": {"type": "integer"},
                    "score": {"type": "integer", "description": "0-100 virality as a stand-alone clip"},
                    "title": {"type": "string"},
                    "reason": {"type": "string", "description": "one sentence on why it works or doesn't"},
                    "start": {"type": "number", "description": "seconds from the start of the video"},
                    "end": {"type": "number", "description": "seconds from the start of the video"},
                },
                "required": ["id", "score", "title", "reason", "start", "end"],
                "additionalProperties": False,
            },
        }
    },
    "required": ["clips"],
    "additionalProperties": False,
}
FALLBACK_CAPABLE = {"claude-opus-5-5", "claude-opus-5", "claude-fable-5-1", "claude-sonnet-5-5"}


def build_prompt(pool):
    parts = [f"Video title: {META.get('title') or 'unknown'}\nVideo length: {fmt_ts(DURATION)}\n",
             f"Below are {len(pool)} candidate clips with their transcripts (timestamps are seconds from the start of the video) "
             "and notes from automatic audio/visual analysis. Score every candidate.\n"]
    for k, c in enumerate(pool):
        lines = "\n".join(f"[{s['start']:.1f}] {s['text']}" for s in clip_segments(c))
        parts.append(f'<candidate id="{k}" start="{c["start"]:.1f}" end="{c["end"]:.1f}" length="{c["end"] - c["start"]:.0f}s">\n'
                     f"<signals>{c['why']}</signals>\n<transcript>\n{lines}\n</transcript>\n</candidate>")
    return "\n".join(parts)


def snap_inside(c, new_start, new_end):
    """Snap Claude's trim to segment boundaries inside the original window; keep the original if invalid."""
    segs = clip_segments(c)
    if not segs:
        return c["start"], c["end"]
    starts = [s["start"] for s in segs if s["start"] >= c["start"] - 0.5]
    ends = [s["end"] for s in segs if s["end"] <= c["end"] + 0.5]
    if not starts or not ends:
        return c["start"], c["end"]
    s = min(starts, key=lambda x: abs(x - new_start))
    e = min(ends, key=lambda x: abs(x - new_end))
    s, e = max(c["start"], s - 0.25), min(c["end"], e + 0.6)
    if e - s < MIN_CLIP_SEC:
        return c["start"], c["end"]
    return s, e


CLAUDE_USED = False
key = get_api_key() if USE_CLAUDE_RANKING else None
if not USE_CLAUDE_RANKING:
    print("Claude ranking turned off in settings - using signal ranking.")
elif not key:
    print("No ANTHROPIC_API_KEY found in Colab Secrets - using signal ranking only.")
elif not SEGMENTS:
    print("No transcript to judge - using signal ranking only.")
else:
    import anthropic
    client = anthropic.Anthropic(api_key=key)
    request = dict(
        model=CLAUDE_MODEL,
        max_tokens=16000,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": build_prompt(POOL)}],
        output_config={"effort": "medium", "format": {"type": "json_schema", "schema": CLAUDE_SCHEMA}},
    )
    print(f"Asking Claude to judge {len(POOL)} candidates...")
    try:
        if CLAUDE_MODEL in FALLBACK_CAPABLE:
            # If a safety classifier declines, retry server-side on Anthropic's recommended fallback model.
            resp = client.beta.messages.create(betas=["server-side-fallback-2026-07-01"], fallbacks="default", **request)
        else:
            resp = client.messages.create(**request)
        if resp.stop_reason == "refusal":
            print("Claude declined to rank these clips - using signal ranking only.")
        elif resp.stop_reason == "max_tokens":
            print("Claude's answer was cut off - using signal ranking only.")
        else:
            data = json.loads(next(b.text for b in resp.content if b.type == "text"))
            for r in data["clips"]:
                if not 0 <= r["id"] < len(POOL):
                    continue
                c = POOL[r["id"]]
                c["claude"] = max(0, min(100, int(r["score"])))
                c["title"] = r["title"].strip()[:100] or c["title"]
                c["claude_reason"] = r["reason"].strip()
                c["start"], c["end"] = snap_inside(c, float(r["start"]), float(r["end"]))
            scored = [c for c in POOL if "claude" in c]
            CLAUDE_USED = bool(scored)
            print(f"Claude scored {len(scored)}/{len(POOL)} candidates.")
    except anthropic.AuthenticationError:
        print("The ANTHROPIC_API_KEY was rejected - check the secret. Using signal ranking only.")
    except anthropic.RateLimitError:
        print("Rate-limited by the Claude API - wait a minute and re-run this cell, or continue with signal ranking.")
    except anthropic.APIStatusError as e:
        print(f"Claude API error {e.status_code}: {e.message} - using signal ranking only.")
    except anthropic.APIConnectionError:
        print("Couldn't reach the Claude API - using signal ranking only.")

if CLAUDE_USED:
    sig = np.array([c["signal"] for c in POOL])
    sig = (sig - sig.min()) / max(np.ptp(sig), 1e-9)
    for c, s in zip(POOL, sig):
        c["final"] = 0.35 * float(s) + 0.65 * c.get("claude", 0) / 100

FINAL = pick_non_overlapping(POOL, NUM_CLIPS, key=lambda c: c["final"])
print(f"\nFinal top {len(FINAL)}:")
for rank, c in enumerate(FINAL, 1):
    extra = f"  claude {c['claude']}" if "claude" in c else ""
    print(f"{rank:2d}. [{fmt_ts(c['start'])}-{fmt_ts(c['end'])}] score {c['final']:.2f}{extra}  {c['title']}")

## 10 · Render the clips
Makes 1920×1080 H.264/AAC MP4s with short fades at each end. For every clip it also writes an `.srt` caption file, plus a `report.md` and `report.csv` listing the ranking, timestamps, titles and the reason each clip was picked. Everything goes into `MyDrive/viral_clips/<video title>_<date>/`.

In [ ]:
#@title 10 · Render the clips
if "FINAL" not in globals() or not FINAL:
    raise RuntimeError("No clips selected - run Cells 8 and 9 first.")


def slug(text, n=50):
    s = re.sub(r"[^\w\s-]", "", text, flags=re.UNICODE).strip().lower()
    return re.sub(r"[\s_-]+", "_", s)[:n].strip("_") or "clip"


def srt_time(t):
    ms = int(round(max(0.0, t) * 1000))
    return f"{ms // 3600000:02d}:{ms % 3600000 // 60000:02d}:{ms % 60000 // 1000:02d},{ms % 1000:03d}"


def build_srt(c):
    """Caption cues (max ~42 chars / 3.5 s each) from word timestamps, shifted to clip time."""
    s0, s1 = c["start"], c["end"]
    words = [w for seg in clip_segments(c) for w in (seg["words"] or [[seg["start"], seg["end"], seg["text"]]])
             if s0 <= (w[0] + w[1]) / 2 <= s1 and w[2]]
    cues, cur = [], []

    def flush():
        if cur:
            cues.append((max(0.0, cur[0][0] - s0), min(s1 - s0, cur[-1][1] - s0), " ".join(w[2] for w in cur)))
            cur.clear()

    for w in words:
        if cur and (len(" ".join(x[2] for x in cur + [w])) > 42 or w[1] - cur[0][0] > 3.5
                    or w[0] - cur[-1][1] > 0.8 or cur[-1][2].endswith(SENT_END)):
            flush()
        cur.append(w)
    flush()
    return "\n".join(f"{k}\n{srt_time(a)} --> {srt_time(max(b, a + 0.3))}\n{t}\n" for k, (a, b, t) in enumerate(cues, 1))


if FIT_MODE == "crop":
    FIT = "scale=1920:1080:force_original_aspect_ratio=increase,crop=1920:1080"
else:
    FIT = "scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2:black"

RUN_DIR = OUTPUT_ROOT / f"{slug(META.get('title') or 'video', 40)}_{time.strftime('%Y%m%d-%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
render_dir = WORK / "render"
shutil.rmtree(render_dir, ignore_errors=True)
render_dir.mkdir(parents=True)

made = []
for rank, c in enumerate(FINAL, 1):
    L = c["end"] - c["start"]
    name = f"{rank:02d}_{slug(c['title'])}"
    srt_text = build_srt(c) if SEGMENTS else ""
    srt_path = render_dir / f"clip{rank:02d}.srt"
    srt_path.write_text(srt_text, encoding="utf-8")

    vf = [FIT, "setsar=1", "fade=t=in:st=0:d=0.3", f"fade=t=out:st={max(0, L - 0.4):.2f}:d=0.4"]
    if BURN_CAPTIONS and srt_text:
        vf.append(f"subtitles={srt_path}:force_style='FontName=DejaVu Sans,FontSize=16,Outline=2,Shadow=0,MarginV=24'")
    out_tmp = render_dir / f"clip{rank:02d}.mp4"
    print(f"Rendering {rank:02d}/{len(FINAL)}  {fmt_ts(c['start'])}-{fmt_ts(c['end'])}  {c['title']}")
    res = run(["ffmpeg", "-y", "-v", "error", "-ss", f"{c['start']:.3f}", "-i", SOURCE, "-t", f"{L:.3f}",
               "-vf", ",".join(vf), "-af", f"afade=t=in:st=0:d=0.3,afade=t=out:st={max(0, L - 0.4):.2f}:d=0.4",
               "-c:v", "libx264", "-preset", "fast", "-crf", "20", "-pix_fmt", "yuv420p",
               "-c:a", "aac", "-b:a", "192k", "-movflags", "+faststart", out_tmp], check=False)
    if res.returncode != 0 or not out_tmp.exists():
        print(f"   failed: {res.stderr.strip()[-400:]}")
        continue
    shutil.copy(out_tmp, RUN_DIR / f"{name}.mp4")
    if srt_text:
        shutil.copy(srt_path, RUN_DIR / f"{name}.srt")
    made.append((rank, name, c))

# ---- reports -----------------------------------------------------------------------
link = META.get("webpage_url")
with open(RUN_DIR / "report.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["rank", "file", "title", "start", "end", "length_sec", "final_score", "signal_score", "claude_score",
                "why", "claude_reason", "source_link"])
    for rank, name, c in made:
        w.writerow([rank, f"{name}.mp4", c["title"], fmt_ts(c["start"]), fmt_ts(c["end"]), round(c["end"] - c["start"], 1),
                    round(c["final"], 3), round(c["signal"], 3), c.get("claude", ""), c["why"], c.get("claude_reason", ""),
                    f"{link}&t={int(c['start'])}s" if link and "watch?v=" in link else (link or "")])

lines = [f"# {META.get('title') or 'Video'} - top {len(made)} moments", ""]
if link:
    lines += [f"Source: {link}", ""]
lines += [f"Ranked by {'Claude + audio/visual/transcript signals' if CLAUDE_USED else 'audio/visual/transcript signals'}.", ""]
for rank, name, c in made:
    lines += [f"## {rank}. {c['title']}",
              f"- **File:** `{name}.mp4` ({c['end'] - c['start']:.0f}s)",
              f"- **From:** {fmt_ts(c['start'])} to {fmt_ts(c['end'])}"
              + (f" ([open in YouTube]({link}&t={int(c['start'])}s))" if link and "watch?v=" in link else ""),
              f"- **Score:** {c['final']:.2f}" + (f" (Claude {c['claude']}/100)" if "claude" in c else ""),
              f"- **Signals:** {c['why']}"]
    if c.get("claude_reason"):
        lines.append(f"- **Why it works:** {c['claude_reason']}")
    lines.append("")
(RUN_DIR / "report.md").write_text("\n".join(lines), encoding="utf-8")
(RUN_DIR / "transcript.json").write_text(json.dumps(SEGMENTS, ensure_ascii=False), encoding="utf-8")

print(f"\nDone: {len(made)}/{len(FINAL)} clips saved to {RUN_DIR}")

## 11 · Check the results

In [ ]:
#@title 11 · Check the results
if "RUN_DIR" not in globals():
    raise RuntimeError("Run Cell 10 first.")
for p in sorted(RUN_DIR.iterdir()):
    print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.name}")
print()
print((RUN_DIR / "report.md").read_text(encoding="utf-8"))